# HAMR untuk deteksi sarkasme Twitter Indonesia

Notebook ini membandingkan baseline, weighted loss, dan HAMR memakai split train yang sama. Model klasifikasi: `indobenchmark/indobert-base-p1`. HAMR memakai meta-set terpisah yang diambil dari train; validation hanya untuk pemilihan checkpoint, dan test hanya untuk evaluasi akhir.

Jalankan di Colab dengan GPU aktif. Output eksperimen disimpan ke Google Drive agar run bisa dilanjutkan setelah runtime terputus. HAMR dan IdSarcasm di-clone terpisah karena folder HAMR lokal belum menjadi bagian dari repo IdSarcasm.

Encoder KNN awal: [paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), model multilingual dengan embedding dense 384 dimensi.

## 1. Periksa runtime GPU

Pilih Runtime → Change runtime type → GPU sebelum melanjutkan. Notebook berhenti bila GPU tidak tersedia.

In [1]:
import torch
assert torch.cuda.is_available(), "Aktifkan GPU Colab melalui Runtime > Change runtime type > GPU."
props = torch.cuda.get_device_properties(0)
print("PyTorch:", torch.__version__)
print("GPU:", props.name)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")

PyTorch: 2.11.0+cu130
GPU: Tesla T4
VRAM: 14.6 GiB


## 2. Instal dependensi dan clone repositori

PyTorch bawaan Colab dipertahankan. Bila Colab memerlukan restart setelah instalasi paket, restart runtime lalu jalankan ulang notebook dari awal.

In [2]:
%pip -q install "transformers==4.47.1" datasets evaluate accelerate sentence-transformers faiss-cpu scikit-learn

In [3]:
from pathlib import Path
import subprocess, os

ID_REPO = Path("/content/id_sarcasm")
HAMR_REPO = Path("/content/ImbalanceLearning")

if not ID_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/w11wo/id_sarcasm.git", str(ID_REPO)], check=True)
if not HAMR_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/trust-nlp/ImbalanceLearning.git", str(HAMR_REPO)], check=True)

assert (ID_REPO / "scripts/run_classification.py").exists()
assert (HAMR_REPO / "HAMR/CLS/train_cls.py").exists()
print("IdSarcasm:", ID_REPO)
print("HAMR:", HAMR_REPO)

IdSarcasm: /content/id_sarcasm
HAMR: /content/ImbalanceLearning


## 3. Mount Drive dan konfigurasi eksperimen

In [4]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, hashlib, gc, shutil, time

ROOT_OUT = Path("/content/drive/MyDrive/idsarcasm/hamr_twitter")
DATA_DIR = ROOT_OUT / "data"
RUN_DIR = ROOT_OUT / "runs"
EMBED_DIR = ROOT_OUT / "embeddings"
for path in (DATA_DIR, RUN_DIR, EMBED_DIR):
    path.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "w11wo/twitter_indonesia_sarcastic"
TEXT_COLUMN = "tweet"
MODEL_NAME = "indobenchmark/indobert-base-p1"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

MAX_LENGTH = 128
TRAIN_BATCH = 8
GRAD_ACCUM = 4
EVAL_BATCH = 16
LEARNING_RATE = 1e-5
EPOCHS = 20
SEEDS = [42, 43, 44]
META_SEED = 42
META_FRACTION = 0.10

print("Output Drive:", ROOT_OUT)
print("Effective batch size:", TRAIN_BATCH * GRAD_ACCUM)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output Drive: /content/drive/MyDrive/idsarcasm/hamr_twitter
Effective batch size: 32


## 4. Muat data, sisihkan meta-set, dan ekspor split

Meta-set diambil secara stratified dari train, lalu dibuat seimbang dengan downsampling kelas mayoritas. Semua contoh pada meta pool dikeluarkan dari train utama agar tidak ada tumpang tindih. Validation dan test asli tidak diubah.

In [5]:
from datasets import load_dataset, ClassLabel, concatenate_datasets
from collections import Counter

raw = load_dataset(DATASET_NAME)
required_splits = {"train", "validation", "test"}
assert required_splits.issubset(raw.keys()), f"Split tidak lengkap: {raw.keys()}"
for split_name in required_splits:
    assert TEXT_COLUMN in raw[split_name].column_names, f"Kolom {TEXT_COLUMN} tidak ada di {split_name}"
    assert "label" in raw[split_name].column_names

# Pastikan stratified split dapat menggunakan fitur label kelas.
for split_name in required_splits:
    feature = raw[split_name].features["label"]
    if not isinstance(feature, ClassLabel):
        raw[split_name] = raw[split_name].cast_column(
            "label", ClassLabel(names=["non-sarcastic", "sarcastic"])
        )

for split_name in ("train", "validation", "test"):
    labels = set(int(x) for x in raw[split_name]["label"])
    assert labels == {0, 1}, (split_name, labels)
    print(split_name, len(raw[split_name]), Counter(raw[split_name]["label"]))

raw["train"] = raw["train"].add_column("_row_id", list(range(len(raw["train"]))))
partition = raw["train"].train_test_split(
    test_size=META_FRACTION, seed=META_SEED, stratify_by_column="label"
)
train_core = partition["train"]
meta_pool = partition["test"]
class_ids = (0, 1)
n_each = min(sum(int(y) == c for y in meta_pool["label"]) for c in class_ids)
meta_parts = [
    meta_pool.filter(lambda row, c=c: int(row["label"]) == c)
             .shuffle(seed=META_SEED + c)
             .select(range(n_each))
    for c in class_ids
]
meta_set = concatenate_datasets(meta_parts).shuffle(seed=META_SEED)

def write_split(dataset, filename):
    rows = [
        {TEXT_COLUMN: str(text), "label": int(label)}
        for text, label in zip(dataset[TEXT_COLUMN], dataset["label"])
    ]
    path = DATA_DIR / filename
    path.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
    return path

split_paths = {
    "train": write_split(train_core, "train.json"),
    "meta": write_split(meta_set, "meta.json"),
    "validation": write_split(raw["validation"], "validation.json"),
    "test": write_split(raw["test"], "test.json"),
}
assert len(train_core) + len(meta_pool) == len(raw["train"])
assert not set(train_core["_row_id"]).intersection(set(meta_pool["_row_id"]))
assert sum(int(y) == 0 for y in meta_set["label"]) == sum(int(y) == 1 for y in meta_set["label"])
manifest = {
    "dataset": DATASET_NAME, "text_column": TEXT_COLUMN,
    "meta_seed": META_SEED, "meta_fraction": META_FRACTION,
    "split_sizes": {k: len(v) for k, v in {
        "train": train_core, "meta": meta_set,
        "validation": raw["validation"], "test": raw["test"]
    }.items()},
    "label_counts": {k: dict(Counter(int(x) for x in v["label"])) for k, v in {
        "train": train_core, "meta": meta_set,
        "validation": raw["validation"], "test": raw["test"]
    }.items()},
}
(DATA_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.csv:   0%|          | 0.00/226k [00:00<?, ?B/s]

data/validation.csv:   0%|          | 0.00/32.6k [00:00<?, ?B/s]

data/test.csv:   0%|          | 0.00/61.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1878 [00:00<?, ? examples/s]

... (5 outputs left out)

train 1878 Counter({0: 1408, 1: 470})
validation 268 Counter({0: 201, 1: 67})
test 538 Counter({0: 404, 1: 134})


Filter:   0%|          | 0/188 [00:00<?, ? examples/s]

Filter:   0%|          | 0/188 [00:00<?, ? examples/s]

{
  "dataset": "w11wo/twitter_indonesia_sarcastic",
  "text_column": "tweet",
  "meta_seed": 42,
  "meta_fraction": 0.1,
  "split_sizes": {
    "train": 1690,
    "meta": 94,
    "validation": 268,
    "test": 538
  },
  "label_counts": {
    "train": {
      "1": 423,
      "0": 1267
    },
    "meta": {
      "1": 47,
      "0": 47
    },
    "validation": {
      "1": 67,
      "0": 201
    },
    "test": {
      "1": 134,
      "0": 404
    }
  }
}


## 5. Buat embedding KNN

Embedding hanya dibuat dari train utama, dengan urutan yang sama seperti `train.json`. Manifest hash mencegah pemakaian ulang embedding yang tidak cocok dengan data.

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

train_rows = json.loads(split_paths["train"].read_text(encoding="utf-8"))
train_texts = [row[TEXT_COLUMN] for row in train_rows]
train_hash = hashlib.sha256("\n".join(train_texts).encode("utf-8")).hexdigest()
embedding_path = EMBED_DIR / "train_embeddings.npy"
embedding_manifest_path = EMBED_DIR / "manifest.json"

reuse = False
if embedding_path.exists() and embedding_manifest_path.exists():
    old_manifest = json.loads(embedding_manifest_path.read_text(encoding="utf-8"))
    reuse = old_manifest.get("train_text_sha256") == train_hash and old_manifest.get("model") == EMBED_MODEL

if not reuse:
    embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
    embeddings = embedder.encode(
        train_texts, batch_size=32, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    np.save(embedding_path, embeddings)
    embedding_manifest_path.write_text(json.dumps({
        "model": EMBED_MODEL, "train_text_sha256": train_hash,
        "rows": int(embeddings.shape[0]), "dimension": int(embeddings.shape[1])
    }, indent=2), encoding="utf-8")
    del embedder, embeddings
    gc.collect()
    torch.cuda.empty_cache()

embeddings = np.load(embedding_path, mmap_mode="r")
assert embeddings.shape == (len(train_rows), 384), embeddings.shape
print("Embedding shape:", embeddings.shape, "| reused:", reuse)
del embeddings
gc.collect()
torch.cuda.empty_cache()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Embedding shape: (1690, 384) | reused: False


## 6. Terapkan patch kompatibilitas pada clone Colab

Patch ini hanya mengubah file di `/content`, bukan repo lokal. HAMR menerima file meta terpisah, mempertahankan indeks contoh untuk menyelaraskan embedding, memakai early stopping, dan menyesuaikan signature Trainer untuk Transformers 4.47.1. Test labels dipertahankan agar metrik test dihitung setelah model terbaik dipilih.

In [7]:
def replace_once(path, old, new, description):
    path = Path(path)
    source = path.read_text(encoding="utf-8")
    count = source.count(old)
    if count == 0:
        if new in source:
            print("Sudah diterapkan:", description)
            return
        raise RuntimeError(f"Patch tidak menemukan target: {description} ({path})")
    if count != 1:
        raise RuntimeError(f"Target patch tidak unik: {description}, ditemukan {count} kali")
    path.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("Diterapkan:", description)

# IdSarcasm: metrics per kelas, compatibility, validation-only selection, labeled test prediction.
baseline_path = ID_REPO / "scripts/run_classification.py"
replace_once(
    baseline_path,
    "from sklearn.utils.class_weight import compute_class_weight",
    "from sklearn.utils.class_weight import compute_class_weight\nfrom sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support",
    "import metrik klasifikasi"
)
baseline_src = baseline_path.read_text(encoding="utf-8")
if "classes=classes, y=" in baseline_src:
    replace_once(baseline_path, "classes=classes, y=", "classes=np.array(classes), y=", "compatibilitas compute_class_weight")
replace_once(
    baseline_path,
    "def compute_loss(self, model, inputs, return_outputs=False):",
    "def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):",
    "signature compute_loss"
)
baseline_src = baseline_path.read_text(encoding="utf-8")
metric_start = baseline_src.index("    def compute_metrics(p: EvalPrediction):")
metric_end = baseline_src.index("    # Data collator", metric_start)
metric_impl = '''    def compute_metrics(p: EvalPrediction):
        logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        preds = np.argmax(logits, axis=1)
        labels = np.asarray(p.label_ids)
        precision_by_class, recall_by_class, f1_by_class, _ = precision_recall_fscore_support(
            labels, preds, labels=[0, 1], zero_division=0
        )
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
            "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
            "precision_nonsarcastic": float(precision_by_class[0]),
            "recall_nonsarcastic": float(recall_by_class[0]),
            "f1_nonsarcastic": float(f1_by_class[0]),
            "precision_sarcastic": float(precision_by_class[1]),
            "recall_sarcastic": float(recall_by_class[1]),
            "f1_sarcastic": float(f1_by_class[1]),
        }

'''
baseline_path.write_text(baseline_src[:metric_start] + metric_impl + baseline_src[metric_end:], encoding="utf-8")
replace_once(
    baseline_path,
    "metrics = trainer.evaluate(eval_dataset=predict_dataset)",
    "metrics = trainer.evaluate(eval_dataset=eval_dataset)",
    "evaluasi validation, bukan test"
)
replace_once(
    baseline_path,
    "        # Removing the `label` columns if exists because it might contains -1 and Trainer won't like that.\n        if \"label\" in predict_dataset.features:\n            predict_dataset = predict_dataset.remove_columns(\"label\")\n",
    "        # Keep test labels so trainer.predict reports final test metrics after checkpoint selection.\n",
    "pertahankan label test untuk evaluasi akhir"
)

# The checkpoint config reports five labels; explicitly set the binary head from this dataset's labels.
BINARY_CONFIG_OLD = '    if is_regression:\n        config.problem_type = "regression"\n'
BINARY_CONFIG_NEW = (
    '    if not is_regression and not is_multi_label:\n'
    '        if len(label_list) != 2:\n'
    '            raise ValueError(f"Expected binary labels, got {label_list}")\n'
    '        config.num_labels = 2\n'
    '        config.id2label = {0: str(label_list[0]), 1: str(label_list[1])}\n'
    '        config.label2id = {str(label_list[0]): 0, str(label_list[1]): 1}\n\n'
    + BINARY_CONFIG_OLD
)
replace_once(baseline_path, BINARY_CONFIG_OLD, BINARY_CONFIG_NEW, "pastikan classifier baseline memiliki 2 label")

# HAMR: add a separate meta JSON input and early stopping.
hamr_train_path = HAMR_REPO / "HAMR/CLS/train_cls.py"
replace_once(
    hamr_train_path,
    "    test_file: Optional[str] = field(default=None, metadata={\"help\": \"A csv or a json file containing the test data.\"})",
    "    test_file: Optional[str] = field(default=None, metadata={\"help\": \"A csv or a json file containing the test data.\"})\n    meta_file: Optional[str] = field(default=None, metadata={\"help\": \"A separate meta-training JSON/CSV file.\"})",
    "argumen meta_file"
)
replace_once(
    hamr_train_path,
    "    DataCollatorWithPadding,\n    EvalPrediction,",
    "    DataCollatorWithPadding,\n    EarlyStoppingCallback,\n    EvalPrediction,",
    "import early stopping"
)
replace_once(
    hamr_train_path,
    '        data_files = {"train": data_args.train_file, "validation": data_args.validation_file}\n',
    '        data_files = {"train": data_args.train_file, "validation": data_args.validation_file}\n        if data_args.meta_file is not None:\n            data_files["meta"] = data_args.meta_file\n',
    "muat meta split"
)
replace_once(
    hamr_train_path,
    "    # Record original indices for HardnessAwareSampler\n",
    '''    # Keep only model inputs and labels; custom training_step separately consumes orig_idx.
    for split_name in raw_datasets.keys():
        keep_columns = {"input_ids", "attention_mask", "token_type_ids", "label"}
        drop_columns = [column for column in raw_datasets[split_name].column_names if column not in keep_columns]
        if drop_columns:
            raw_datasets[split_name] = raw_datasets[split_name].remove_columns(drop_columns)

    # Record original indices for HardnessAwareSampler
''',
    "kolom batch HAMR"
)
replace_once(
    hamr_train_path,
    '''    # Use validation set as meta-dataset
    meta_dataset = None
    if training_args.hardness_aware_sampling:
        if not training_args.do_eval:
            raise ValueError("When hardness_aware_sampling=true, a validation set (do_eval=true) must be provided as meta-set.")
        meta_dataset = eval_dataset
        logger.info(f"Using validation set (size: {len(meta_dataset)}) as meta-set.")
''',
    '''    # Use a separate held-out training meta-set; validation remains for checkpoint selection.
    meta_dataset = None
    if training_args.hardness_aware_sampling:
        if "meta" not in raw_datasets:
            raise ValueError("--meta_file is required when hardness_aware_sampling=true.")
        meta_dataset = raw_datasets["meta"]
        logger.info(f"Using separate meta set (size: {len(meta_dataset)})")
''',
    "pisahkan meta-set dari validation"
)
replace_once(
    hamr_train_path,
    "    # Enable KNN neighbor-boost update at the end of each epoch.\n",
    "    trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01))\n\n    # Enable KNN neighbor-boost update at the end of each epoch.\n",
    "early stopping HAMR"
)
# Replace HAMR metrics with the same binary-class metrics as the baseline.
hamr_src = hamr_train_path.read_text(encoding="utf-8")
if "from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support" not in hamr_src:
    replace_once(
        hamr_train_path,
        "import numpy as np\n",
        "import numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support\n",
        "import metrik HAMR"
    )
hamr_src = hamr_train_path.read_text(encoding="utf-8")
metric_start = hamr_src.index("    def compute_metrics(p: EvalPrediction):")
metric_end = hamr_src.index("    # Data collator", metric_start)
hamr_metric_impl = '''    def compute_metrics(p: EvalPrediction):
        logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        preds = np.argmax(logits, axis=1)
        labels = np.asarray(p.label_ids)
        precision_by_class, recall_by_class, f1_by_class, _ = precision_recall_fscore_support(
            labels, preds, labels=[0, 1], zero_division=0
        )
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
            "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
            "precision_nonsarcastic": float(precision_by_class[0]),
            "recall_nonsarcastic": float(recall_by_class[0]),
            "f1_nonsarcastic": float(f1_by_class[0]),
            "precision_sarcastic": float(precision_by_class[1]),
            "recall_sarcastic": float(recall_by_class[1]),
            "f1_sarcastic": float(f1_by_class[1]),
        }

'''
hamr_train_path.write_text(hamr_src[:metric_start] + hamr_metric_impl + hamr_src[metric_end:], encoding="utf-8")

replace_once(hamr_train_path, BINARY_CONFIG_OLD, BINARY_CONFIG_NEW, "pastikan classifier HAMR memiliki 2 label")

hamr_trainer_path = HAMR_REPO / "HAMR/CLS/custom_trainer.py"
replace_once(
    hamr_trainer_path,
    "    def _get_train_sampler(\n        self,\n        dataset,  # The current train_dataset passed by transformers.Trainer\n    ) -> Optional[torch.utils.data.Sampler]:\n",
    "    def _get_train_sampler(\n        self,\n        dataset=None,\n    ) -> Optional[torch.utils.data.Sampler]:\n        if dataset is None:\n            dataset = self.train_dataset\n",
    "kompatibilitas signature sampler Transformers"
)
replace_once(
    hamr_trainer_path,
    "return super()._get_train_sampler(dataset)",
    "return super()._get_train_sampler()",
    "kompatibilitas fallback sampler Transformers"
)
replace_once(
    hamr_trainer_path,
    "    def compute_loss(self, model, inputs, return_outputs=False):",
    "    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):",
    "signature compute_loss HAMR"
)
replace_once(
    hamr_trainer_path,
    "        num_total_examples: int | None = None,",
    "        num_items_in_batch: int | None = None,",
    "signature training_step HAMR"
)
print("Patch runtime selesai. Repo sumber asli di luar /content tidak disentuh.")

Diterapkan: import metrik klasifikasi
Diterapkan: compatibilitas compute_class_weight
Diterapkan: signature compute_loss
Diterapkan: evaluasi validation, bukan test
Diterapkan: pertahankan label test untuk evaluasi akhir
Diterapkan: pastikan classifier baseline memiliki 2 label
Diterapkan: argumen meta_file
Diterapkan: import early stopping
Diterapkan: muat meta split
Diterapkan: kolom batch HAMR
Diterapkan: pisahkan meta-set dari validation
Diterapkan: early stopping HAMR
Diterapkan: import metrik HAMR
Diterapkan: pastikan classifier HAMR memiliki 2 label
Diterapkan: kompatibilitas signature sampler Transformers
Diterapkan: kompatibilitas fallback sampler Transformers
Diterapkan: signature compute_loss HAMR
Diterapkan: signature training_step HAMR
Patch runtime selesai. Repo sumber asli di luar /content tidak disentuh.


## 7. Jalankan smoke run HAMR

Smoke run memakai seluruh train core selama satu epoch dan hanya mengevaluasi validation. Tidak menjalankan prediksi test. Ini memastikan trainer, meta-set, dan embedding tersambung sebelum eksperimen penuh.

In [8]:
import sys

BASE_ARGS = [
    "--model_name_or_path", MODEL_NAME,
    "--train_file", str(DATA_DIR / "train.json"),
    "--validation_file", str(DATA_DIR / "validation.json"),
    "--test_file", str(DATA_DIR / "test.json"),
    "--text_column_names", TEXT_COLUMN,
    "--label_column_name", "label",
    "--max_seq_length", str(MAX_LENGTH),
    "--per_device_train_batch_size", str(TRAIN_BATCH),
    "--gradient_accumulation_steps", str(GRAD_ACCUM),
    "--per_device_eval_batch_size", str(EVAL_BATCH),
    "--learning_rate", str(LEARNING_RATE),
    "--lr_scheduler_type", "cosine",
    "--weight_decay", "0.03",
    "--num_train_epochs", str(EPOCHS),
    "--save_strategy", "epoch",
    "--evaluation_strategy", "epoch",
    "--logging_strategy", "epoch",
    "--load_best_model_at_end",
    "--metric_for_best_model", "f1_macro",
    "--greater_is_better", "true",
    "--report_to", "none",
    "--fp16",
    "--do_train", "--do_eval",
]
HAMR_ARGS = [
    "--meta_file", str(DATA_DIR / "meta.json"),
    "--hardness_aware_sampling", "true",
    "--hardness_alpha", "0.7",
    "--knn_k", "10",
    "--knn_lambda", "0.1",
    "--knn_build_freq", "1",
    "--knn_hard_sample_ratio", "0.1",
    "--wnet_lr", "1e-4",
    "--meta_update_lr", "2e-4",
    "--embedding_dir", str(EMBED_DIR),
    "--weighted_loss", "true",
    "--weighted_sampling", "true",
    "--sampler_oversample_ratio", "1.0",
    "--remove_unused_columns", "false",
]
SMOKE_DIR = RUN_DIR / "smoke_hamr"
smoke_args = BASE_ARGS.copy()
# Smoke hanya memvalidasi training/validation, jadi jangan berikan test_file.
test_arg_index = smoke_args.index("--test_file")
del smoke_args[test_arg_index:test_arg_index + 2]
smoke_args[smoke_args.index("--num_train_epochs") + 1] = "1"
smoke_args += ["--seed", "42", "--output_dir", str(SMOKE_DIR)]
smoke_command = [sys.executable, str(HAMR_REPO / "HAMR/CLS/train_cls.py"), *smoke_args, *HAMR_ARGS]
smoke_result = subprocess.run(
    smoke_command, cwd=HAMR_REPO / "HAMR/CLS", text=True, capture_output=True
)
print(smoke_result.stdout)
if smoke_result.stderr:
    print("STDERR:\n", smoke_result.stderr)
if smoke_result.returncode != 0:
    raise RuntimeError(f"HAMR smoke run gagal (exit={smoke_result.returncode}); pesan error lengkap ada di atas.")
print("Smoke run HAMR selesai; periksa eval_results.json:", SMOKE_DIR / "eval_results.json")

{'loss': 0.5108, 'grad_norm': 8.689347267150879, 'learning_rate': 0.0, 'epoch': 1.0}
{'eval_loss': 0.48133838176727295, 'eval_accuracy': 0.7686567164179104, 'eval_f1_macro': 0.553669281186204, 'eval_f1_micro': 0.7686567164179104, 'eval_precision_nonsarcastic': 0.7747035573122529, 'eval_recall_nonsarcastic': 0.9751243781094527, 'eval_f1_nonsarcastic': 0.8634361233480177, 'eval_precision_sarcastic': 0.6666666666666666, 'eval_recall_sarcastic': 0.14925373134328357, 'eval_f1_sarcastic': 0.24390243902439024, 'eval_runtime': 0.9233, 'eval_samples_per_second': 290.278, 'eval_steps_per_second': 18.413, 'epoch': 1.0}
{'train_runtime': 89.8806, 'train_samples_per_second': 18.803, 'train_steps_per_second': 0.59, 'train_loss': 0.5107722012501843, 'epoch': 1.0}
***** train metrics *****
  epoch                    =        1.0
  total_flos               =   103529GF
  train_loss               =     0.5108
  train_runtime            = 0:01:29.88
  train_samples            =       1690
  train_samples

## 8. Jalankan eksperimen penuh

Setiap run memakai train yang sama dan output directory sendiri. Run yang selesai ditandai dengan `DONE.json`; jika runtime terputus di tengah run, jalankan ulang cell ini untuk melanjutkan dari checkpoint terakhir.

In [9]:
from transformers.trainer_utils import get_last_checkpoint

def command_for(method, seed, output_dir):
    args = BASE_ARGS.copy()
    args += ["--seed", str(seed), "--output_dir", str(output_dir),
             "--do_predict", "--save_total_limit", "2"]
    if method == "weighted_loss":
        args += ["--do_weighted_loss", "true"]
    if method == "hamr":
        args += HAMR_ARGS
    script = (
        ID_REPO / "scripts/run_classification.py"
        if method in {"baseline", "weighted_loss"}
        else HAMR_REPO / "HAMR/CLS/train_cls.py"
    )
    cwd = ID_REPO if method in {"baseline", "weighted_loss"} else HAMR_REPO / "HAMR/CLS"
    return [sys.executable, str(script), *args], cwd

METHODS = ["baseline", "weighted_loss", "hamr"]
for seed in SEEDS:
    for method in METHODS:
        output_dir = RUN_DIR / method / f"seed_{seed}"
        output_dir.mkdir(parents=True, exist_ok=True)
        done_file = output_dir / "DONE.json"
        if done_file.exists():
            print("Lewati run selesai:", method, seed)
            continue
        command, cwd = command_for(method, seed, output_dir)
        # Resume when a valid checkpoint exists. Otherwise allow Transformers
        # to create its logging directory inside output_dir before training.
        checkpoint = get_last_checkpoint(str(output_dir))
        if checkpoint is None:
            command.append("--overwrite_output_dir")
            print("Tidak ada checkpoint; mulai dari awal:", method, seed)
        elif checkpoint is not None:
            print("Lanjutkan dari checkpoint:", checkpoint)
        log_file = output_dir / "run.log"
        print(f"Mulai {method}, seed={seed}")
        with log_file.open("a", encoding="utf-8") as log:
            process = subprocess.Popen(
                command, cwd=cwd, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, bufsize=1
            )
            for line in process.stdout:
                print(line, end="")
                log.write(line)
            return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(
                f"Run gagal: {method}, seed={seed}. Periksa log: {log_file}. "
                "Run dapat dilanjutkan dari checkpoint dengan menjalankan ulang cell."
            )
        done_file.write_text(json.dumps({
            "method": method, "seed": seed, "command": command,
            "completed_unix": time.time()
        }, indent=2), encoding="utf-8")
        gc.collect()
        torch.cuda.empty_cache()

Streaming output truncated to the last 5000 lines.
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=True,
do_train=True,
embedding_dir=/content/drive/MyDrive/idsarcasm/hamr_twitter/embeddings,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_strategy=epoch,
fp16=True,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_con

## 9. Evaluasi test sekali per run dan rangkum hasil

Script training memilih checkpoint dari validation, lalu memanggil prediksi test satu kali setelah training. Cell ini membaca metrik yang tersimpan, menambahkan confusion matrix dan classification report dari prediksi berlabel, lalu membuat ringkasan mean ± standard deviation per metode.

In [10]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

test_rows = json.loads((DATA_DIR / "test.json").read_text(encoding="utf-8"))
y_true = np.asarray([int(row["label"]) for row in test_rows])
all_rows = []

for seed in SEEDS:
    for method in METHODS:
        output_dir = RUN_DIR / method / f"seed_{seed}"
        if not (output_dir / "DONE.json").exists():
            print("Belum selesai:", method, seed)
            continue
        pred_file = output_dir / "predict_results.txt"
        lines = pred_file.read_text(encoding="utf-8").strip().splitlines()[1:]
        y_pred = np.asarray([int(line.split("\t", 1)[1]) for line in lines])
        assert len(y_pred) == len(y_true), (method, seed, len(y_pred), len(y_true))
        report = classification_report(
            y_true, y_pred, labels=[0, 1],
            target_names=["non_sarcastic", "sarcastic"],
            output_dict=True, zero_division=0
        )
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
        predict_metrics_path = output_dir / "predict_results.json"
        predict_metrics = json.loads(predict_metrics_path.read_text(encoding="utf-8")) if predict_metrics_path.exists() else {}
        validation_path = output_dir / "eval_results.json"
        validation_metrics = json.loads(validation_path.read_text(encoding="utf-8")) if validation_path.exists() else {}
        result = {
            "method": method, "seed": seed,
            "validation": validation_metrics,
            "test_metrics": predict_metrics,
            "test_confusion_matrix_labels_0_1": cm,
            "test_classification_report": report,
        }
        (output_dir / "final_report.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        all_rows.append({
            "method": method, "seed": seed,
            "test_macro_f1": report["macro avg"]["f1-score"],
            "test_f1_sarcastic": report["sarcastic"]["f1-score"],
            "test_precision_sarcastic": report["sarcastic"]["precision"],
            "test_recall_sarcastic": report["sarcastic"]["recall"],
            "test_f1_nonsarcastic": report["non_sarcastic"]["f1-score"],
            "confusion_matrix": cm,
        })

results = pd.DataFrame(all_rows)
results.to_csv(ROOT_OUT / "test_results_by_seed.csv", index=False)
summary = results.groupby("method").agg(
    runs=("seed", "count"),
    macro_f1_mean=("test_macro_f1", "mean"),
    macro_f1_std=("test_macro_f1", "std"),
    sarcastic_f1_mean=("test_f1_sarcastic", "mean"),
    sarcastic_f1_std=("test_f1_sarcastic", "std"),
    nonsarcastic_f1_mean=("test_f1_nonsarcastic", "mean"),
    nonsarcastic_f1_std=("test_f1_nonsarcastic", "std"),
)
summary.to_csv(ROOT_OUT / "test_summary_mean_std.csv")
display(results)
display(summary)
print("Hasil tersimpan di", ROOT_OUT)

,method,seed,test_macro_f1,test_f1_sarcastic,test_precision_sarcastic,test_recall_sarcastic,test_f1_nonsarcastic,confusion_matrix
0,baseline,42,0.824568,0.733591,0.760000,0.708955,0.915545,"[[374, 30], [39, 95]]"
1,weighted_loss,42,0.830179,0.738956,0.800000,0.686567,0.921403,"[[381, 23], [42, 92]]"
2,hamr,42,0.792225,0.686792,0.694656,0.679104,0.897657,"[[364, 40], [43, 91]]"
3,baseline,43,0.813030,0.720588,0.710145,0.731343,0.905473,"[[364, 40], [36, 98]]"
4,weighted_loss,43,0.801280,0.689362,0.801980,0.604478,0.913199,"[[384, 20], [53, 81]]"
5,hamr,43,0.778138,0.652361,0.767677,0.567164,0.903915,"[[381, 23], [58, 76]]"
6,baseline,44,0.799745,0.691057,0.758929,0.634328,0.908434,"[[377, 27], [49, 85]]"
7,weighted_loss,44,0.812416,0.712000,0.767241,0.664179,0.912833,"[[377, 27], [45, 89]]"
8,hamr,44,0.789206,0.674797,0.741071,0.619403,0.903614,"[[375, 29], [51, 83]]"


,runs,macro_f1_mean,macro_f1_std,sarcastic_f1_mean,sarcastic_f1_std,nonsarcastic_f1_mean,nonsarcastic_f1_std
method,,,,,,,
baseline,3,0.812448,0.012421,0.715079,0.021796,0.909817,0.005177
hamr,3,0.786523,0.007417,0.671317,0.017478,0.901729,0.003529
weighted_loss,3,0.814625,0.014576,0.713439,0.024828,0.915811,0.004846


Hasil tersimpan di /content/drive/MyDrive/idsarcasm/hamr_twitter


## Kriteria interpretasi

Fokus pada validation macro-F1 untuk pemilihan checkpoint. Setelah semua konfigurasi terkunci, bandingkan test macro-F1 dan F1 sarkastik antar-seed. HAMR layak dipertahankan bila kenaikannya konsisten dan F1 non-sarkastik tidak turun tajam. Simpan hasil sebagai eksperimen awal; ulangi pada Reddit dan/atau beberapa seed tambahan sebelum menarik kesimpulan umum.